In [13]:
import os
import torch
import torch.nn as nn 
import numpy as np
import pyexr
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

In [14]:
class BaselineHolographyDataset(Dataset):
    def __init__(self, root_dir):
        self.img_dir = os.path.join(root_dir, "img")
        self.depth_dir = os.path.join(root_dir, "depth")
        self.amp_dir = os.path.join(root_dir, "amp")
        self.phs_dir = os.path.join(root_dir, "phs")
        
        # Get indices
        self.indices = sorted([f.replace(".exr", "") for f in os.listdir(self.img_dir) if f.endswith(".exr")])

    def load_exr(self, path):
        file = pyexr.open(path)
        return file.get().astype(np.float32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        
        # 1. Load raw RGB and Depth
        rgb_raw = self.load_exr(os.path.join(self.img_dir, f"{i}.exr"))
        depth_raw = self.load_exr(os.path.join(self.depth_dir, f"{i}.exr"))
        
        # 2. Prepare 4-Channel Input [RGB (3) + Depth (1)]
        # RGB to [3, 512, 512]
        rgb = torch.from_numpy(rgb_raw).permute(2, 0, 1) if rgb_raw.ndim == 3 else torch.from_numpy(rgb_raw).unsqueeze(0)
        # Depth to [1, 512, 512]
        depth = torch.from_numpy(depth_raw).unsqueeze(0) if depth_raw.ndim == 2 else torch.from_numpy(depth_raw).permute(2, 0, 1)[:1]
        
        x = torch.cat([rgb, depth], dim=0) # [4, 512, 512]
        
        # 3. Prepare 2-Channel Target [Amp, Phase]
        amp = torch.from_numpy(self.load_exr(os.path.join(self.amp_dir, f"{i}.exr")))
        phs = torch.from_numpy(self.load_exr(os.path.join(self.phs_dir, f"{i}.exr")))
        
        if amp.ndim == 3: amp = amp[:, :, 0]
        if phs.ndim == 3: phs = phs[:, :, 0]
        y = torch.cat([amp.unsqueeze(0), phs.unsqueeze(0)], dim=0) # [2, 512, 512]

        return x, y

In [15]:
# Initialize Baseline Data
data_root = r"C:\Users\Kai Kumano\workspace\CGH-depth\dataset\KOREATECH-CGH-512-3.6Mu"
train_baseline = BaselineHolographyDataset(os.path.join(data_root, "train"))
train_loader = DataLoader(train_baseline, batch_size=1, shuffle=True)

Model architecture

In [16]:

# Change this part
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class SimpleUNetBaseline(nn.Module):
    def __init__(self, in_channels=4, out_channels=2):
        super(SimpleUNetBaseline, self).__init__()

        # Encoder
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))

        # Decoder
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(128, 64)

        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x); x2 = self.down1(x1); x3 = self.down2(x2); x4 = self.down3(x3); x5 = self.down4(x4)
        u1 = self.up1(x5); u1 = torch.cat([u1, x4], dim=1); u1 = self.conv_up1(u1)
        u2 = self.up2(u1); u2 = torch.cat([u2, x3], dim=1); u2 = self.conv_up2(u2)
        u3 = self.up3(u2); u3 = torch.cat([u3, x2], dim=1); u3 = self.conv_up3(u3)
        u4 = self.up4(u3); u4 = torch.cat([u4, x1], dim=1); u4 = self.conv_up4(u4)
        return self.outc(u4)
    
model_baseline = SimpleUNetBaseline(in_channels=4, out_channels=2).to(device)
print(f"Baseline Model initialized on: {device}")

Baseline Model initialized on: cuda


Model training

In [17]:
import torch.optim as optim

# 1. Setup for Baseline
criterion = torch.nn.MSELoss()
optimizer = optim.Adam(model_baseline.parameters(), lr=1e-4)

# Use a scheduler to match your engineered version's behavior
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

num_epochs = 20  # Shorter run for baseline comparison

print(f"Starting Baseline Training on {device}...")

for epoch in range(num_epochs):
    model_baseline.train()
    running_loss = 0.0
    
    # Use the baseline loader (the 4-channel one)
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model_baseline(inputs)
        
        # Calculate loss
        loss = criterion(outputs, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Print progress every 10 batches to match your other loop
        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Batch [{batch_idx}/{len(train_loader)}] Baseline Loss: {loss.item():.6f}")

    # Calculate average epoch loss
    epoch_loss = running_loss / len(train_loader)
    print(f"==> Epoch {epoch+1} Complete. Average Baseline Loss: {epoch_loss:.6f}")
    
    # Update scheduler based on epoch loss
    scheduler.step(epoch_loss)
    
    # Save baseline model periodically
    if (epoch + 1) % 10 == 0:
        torch.save(model_baseline.state_dict(), f"unet_baseline_epoch_{epoch+1}.pth")

print("Baseline Training Finished.")

Starting Baseline Training on cuda...
Epoch [1/20] Batch [0/5000] Baseline Loss: 0.412674
Epoch [1/20] Batch [10/5000] Baseline Loss: 0.152597
Epoch [1/20] Batch [20/5000] Baseline Loss: 0.083155
Epoch [1/20] Batch [30/5000] Baseline Loss: 0.032416
Epoch [1/20] Batch [40/5000] Baseline Loss: 0.020803
Epoch [1/20] Batch [50/5000] Baseline Loss: 0.020129
Epoch [1/20] Batch [60/5000] Baseline Loss: 0.017843
Epoch [1/20] Batch [70/5000] Baseline Loss: 0.016576
Epoch [1/20] Batch [80/5000] Baseline Loss: 0.014382
Epoch [1/20] Batch [90/5000] Baseline Loss: 0.020584
Epoch [1/20] Batch [100/5000] Baseline Loss: 0.016463
Epoch [1/20] Batch [110/5000] Baseline Loss: 0.014886
Epoch [1/20] Batch [120/5000] Baseline Loss: 0.014928
Epoch [1/20] Batch [130/5000] Baseline Loss: 0.015972
Epoch [1/20] Batch [140/5000] Baseline Loss: 0.014426
Epoch [1/20] Batch [150/5000] Baseline Loss: 0.016049
Epoch [1/20] Batch [160/5000] Baseline Loss: 0.012599
Epoch [1/20] Batch [170/5000] Baseline Loss: 0.012471
E

Model prediction

In [18]:
model_path = "unet_baseline_epoch_20.pth"
data_root = r"C:\Users\Kai Kumano\workspace\CGH-depth\dataset\KOREATECH-CGH-512-3.6Mu"
save_dir = r"C:\Users\Kai Kumano\workspace\CGH-depth\results\baseline_predictions"
os.makedirs(save_dir, exist_ok=True)

# 2. LOAD MODEL
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Re-initialize the architecture
model = SimpleUNetBaseline(in_channels=4, out_channels=2).to(device)
# Load the trained weights
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

test_index = "0" # The filename index (e.g., 0.exr)
test_img_path = os.path.join(data_root, "test", "img", f"{test_index}.exr")
test_depth_path = os.path.join(data_root, "test", "depth", f"{test_index}.exr")


In [19]:

# 4. PREPROCESS & PREDICT
def predict_single():
    # Load raw EXR files
    rgb = pyexr.open(test_img_path).get().astype(np.float32)
    depth = pyexr.open(test_depth_path).get().astype(np.float32)

    # Convert to Tensor [1, 4, 512, 512]
    rgb_t = torch.from_numpy(rgb).permute(2, 0, 1)
    depth_t = torch.from_numpy(depth)
    if depth_t.ndim == 3: depth_t = depth_t[:, :, 0]
    
    x = torch.cat([rgb_t, depth_t.unsqueeze(0)], dim=0).unsqueeze(0).to(device)

    # Inference
    with torch.no_grad():
        output = model(x).squeeze(0).cpu().numpy() # [2, 512, 512]
    
    return output

# Run it
prediction = predict_single()
pred_amp = prediction[0]
pred_phs = prediction[1]

# 5. SAVE TO FILE (.exr)
# We store Amp in Red channel and Phase in Green channel for easy retrieval
output_to_save = np.zeros((512, 512, 3), dtype=np.float32)
output_to_save[:, :, 0] = pred_amp
output_to_save[:, :, 1] = pred_phs

save_path = os.path.join(save_dir, f"prediction_{test_index}.exr")
pyexr.write(save_path, output_to_save)
print(f"Success! Prediction saved to: {save_path}")

ExrError: File 'C:\Users\Kai Kumano\workspace\CGH-depth\dataset\KOREATECH-CGH-512-3.6Mu\test\img\0.exr' is not an EXR file.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(pred_amp, cmap='gray')
axes[0].set_title("Predicted Amplitude")
axes[1].imshow(pred_phs, cmap='hsv')
axes[1].set_title("Predicted Phase")
plt.show()